In [ ]:
!pip install matplotlib
!pip install scikit-learn
!pip install torch
!pip install openpyxl
!pip install geopandas

In [ ]:
import os
import ast
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import itertools
import math
import random
import copy

import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

################
### 2024 ###
################

In [ ]:
# =========================================================
# 1. CONFIGURACION GENERAL
# =========================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo en uso: {DEVICE}")

archivo_excel = "ENADES2024.xlsx"
hoja = "BBDD"

In [ ]:
# =========================================================
# 2. LEER ARCHIVO EXCEL
# =========================================================

df = pd.read_excel(archivo_excel, sheet_name=hoja)

print("Dimensiones de la base:", df.shape)
print("Columnas disponibles:")
print(df.columns.tolist())

In [ ]:
# =========================================================
# 3. VARIABLES
# =========================================================

VARIABLES_PERCEPCIONES = [
    "P02a", "P03a", "P05a", "P06a",
    "P07", "P08", "P13",
    "P17", "P18", "P19", "P20", "P21", "P22",
    "P23", "P24", "P25", "P26",
    "P28a", "P29a",
    "P38a"
]

VARIABLES_SOCIODEMOGRAFICAS = [
    "edadr", "sexo", "macrozona", "ur", "hogar", "edu2",
    "ocupa2", "NSE2", "yhogar", "etnicidad2", "ideología2"
]

faltantes_percepciones = [c for c in VARIABLES_PERCEPCIONES if c not in df.columns]
if len(faltantes_percepciones) > 0:
    raise ValueError(f"Faltan variables de percepciones en la base: {faltantes_percepciones}")

faltantes_sociodemograficas = [c for c in VARIABLES_SOCIODEMOGRAFICAS if c not in df.columns]
if len(faltantes_sociodemograficas) > 0:
    print(f"Advertencia: faltan algunas sociodemograficas en la base: {faltantes_sociodemograficas}")

VARIABLES_SOCIODEMOGRAFICAS_EXISTENTES = [c for c in VARIABLES_SOCIODEMOGRAFICAS if c in df.columns]

In [ ]:
# =========================================================
# 4. MATRIZ DE ENTRADA
# =========================================================

X = df[VARIABLES_PERCEPCIONES].copy().astype(float)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val = train_test_split(
    X_scaled,
    test_size=0.20,
    random_state=SEED
)

print("Shape total:", X_scaled.shape)
print("Shape train:", X_train.shape)
print("Shape val:", X_val.shape)

In [ ]:
# =========================================================
# 3. DATASET
# =========================================================

class EncuestaDataset(Dataset):
    def __init__(self, data_array):
        self.data = torch.tensor(data_array, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        return x, x

In [ ]:
# =========================================================
# 4. AUTOENCODER FLEXIBLE
# =========================================================

class AutoencoderFlexible(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim, dropout=0.0, activation_name="relu"):
        super().__init__()

        def build_activation(name):
            if name == "relu":
                return nn.ReLU()
            elif name == "leaky_relu":
                return nn.LeakyReLU(negative_slope=0.1)
            elif name == "elu":
                return nn.ELU()
            else:
                raise ValueError(f"Activacion no soportada: {name}")

        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            encoder_layers.append(nn.Linear(prev_dim, h))
            encoder_layers.append(build_activation(activation_name))
            if dropout > 0:
                encoder_layers.append(nn.Dropout(dropout))
            prev_dim = h
        encoder_layers.append(nn.Linear(prev_dim, latent_dim))
        self.encoder = nn.Sequential(*encoder_layers)

        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        for h in reversed(hidden_dims):
            decoder_layers.append(nn.Linear(prev_dim, h))
            decoder_layers.append(build_activation(activation_name))
            prev_dim = h
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        decoder_layers.append(nn.Sigmoid())
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z


In [ ]:
# =========================================================
# 5. FUNCIONES AUXILIARES
# =========================================================

def get_loss_function(loss_name):
    if loss_name == "mse":
        return nn.MSELoss()
    elif loss_name == "mae":
        return nn.L1Loss()
    elif loss_name == "smoothl1":
        return nn.SmoothL1Loss()
    else:
        raise ValueError(f"Loss no soportada: {loss_name}")

def evaluate_reconstruction(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    total_n = 0

    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            recon, _ = model(xb)
            loss = criterion(recon, yb)

            batch_n = xb.size(0)
            total_loss += loss.item() * batch_n
            total_n += batch_n

    return total_loss / total_n

def extract_latent(model, data_array):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(data_array, dtype=torch.float32).to(DEVICE)
        _, Z = model(X_tensor)
        return Z.cpu().numpy()

def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    max_epochs=400,
    patience=40,
    min_delta=1e-6
):
    best_model_state = None
    best_val_loss = np.inf
    patience_counter = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        train_loss_sum = 0.0
        n_train = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            recon, _ = model(xb)
            loss = criterion(recon, yb)
            loss.backward()
            optimizer.step()

            batch_n = xb.size(0)
            train_loss_sum += loss.item() * batch_n
            n_train += batch_n

        train_loss = train_loss_sum / n_train
        val_loss = evaluate_reconstruction(model, val_loader, criterion)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss
        })

        if val_loss < (best_val_loss - min_delta):
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    history_df = pd.DataFrame(history)
    return model, history_df, best_val_loss

def evaluate_kmeans_on_embedding(Z, k_values=range(2, 7)):
    rows = []

    for k in k_values:
        try:
            kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=50)
            labels = kmeans.fit_predict(Z)

            n_unique = len(np.unique(labels))

            if n_unique < 2:
                rows.append({
                    "k": k,
                    "n_unique_labels": n_unique,
                    "silhouette": np.nan,
                    "davies_bouldin": np.nan,
                    "calinski_harabasz": np.nan,
                    "is_valid": 0
                })
                continue

            silhouette = silhouette_score(Z, labels)
            dbi = davies_bouldin_score(Z, labels)
            chs = calinski_harabasz_score(Z, labels)

            rows.append({
                "k": k,
                "n_unique_labels": n_unique,
                "silhouette": silhouette,
                "davies_bouldin": dbi,
                "calinski_harabasz": chs,
                "is_valid": 1
            })

        except Exception as e:
            rows.append({
                "k": k,
                "n_unique_labels": np.nan,
                "silhouette": np.nan,
                "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan,
                "is_valid": 0,
                "error_message": str(e)
            })

    metrics_df = pd.DataFrame(rows)

    valid_df = metrics_df[metrics_df["is_valid"] == 1].copy()

    if valid_df.empty:
        best_row = None
    else:

        s_min = valid_df["silhouette"].min()
        s_max = valid_df["silhouette"].max()

        if s_max == s_min:
            valid_df["silhouette_norm"] = 1
        else:
            valid_df["silhouette_norm"] = (valid_df["silhouette"] - s_min) / (s_max - s_min)

        db_min = valid_df["davies_bouldin"].min()
        db_max = valid_df["davies_bouldin"].max()

        if db_max == db_min:
            valid_df["davies_bouldin_norm"] = 1
        else:
            valid_df["davies_bouldin_norm"] = 1 - (
                (valid_df["davies_bouldin"] - db_min) / (db_max - db_min)
            )

        ch_min = valid_df["calinski_harabasz"].min()
        ch_max = valid_df["calinski_harabasz"].max()

        if ch_max == ch_min:
            valid_df["calinski_harabasz_norm"] = 1
        else:
            valid_df["calinski_harabasz_norm"] = (
                (valid_df["calinski_harabasz"] - ch_min) / (ch_max - ch_min)
            )

        valid_df["composite_score"] = (
            valid_df["silhouette_norm"]
            + valid_df["davies_bouldin_norm"]
            + valid_df["calinski_harabasz_norm"]
        ) / 3

        idx_best = valid_df["composite_score"].idxmax()
        best_row = valid_df.loc[idx_best].to_dict()

    return metrics_df, best_row

def train_final_model_on_full_data(
    X_full,
    input_dim,
    hidden_dims,
    latent_dim,
    dropout,
    activation_name,
    learning_rate,
    weight_decay,
    batch_size,
    loss_name,
    max_epochs=500
):
    full_dataset = EncuestaDataset(X_full)
    full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True)

    model = AutoencoderFlexible(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        latent_dim=latent_dim,
        dropout=dropout,
        activation_name=activation_name
    ).to(DEVICE)

    criterion = get_loss_function(loss_name)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    history = []

    for epoch in range(max_epochs):
        model.train()
        epoch_loss_sum = 0.0
        total_n = 0

        for xb, yb in full_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            recon, _ = model(xb)
            loss = criterion(recon, yb)
            loss.backward()
            optimizer.step()

            batch_n = xb.size(0)
            epoch_loss_sum += loss.item() * batch_n
            total_n += batch_n

        epoch_loss = epoch_loss_sum / total_n
        history.append({
            "epoch": epoch + 1,
            "full_loss": epoch_loss
        })

    history_df = pd.DataFrame(history)
    return model, history_df

In [ ]:
# =========================================================
# 6. ESPACIO DE BUSQUEDA EXHAUSTIVO
# =========================================================

search_space = {
    "latent_dim": [2, 3, 4, 5, 6, 7, 8, 9, 10],
    "hidden_dims": [
        [16],
        [24],
        [32],
        [48],
        [64],
        [96],
        [16, 8],
        [24, 12],
        [32, 16],
        [48, 24],
        [64, 32],
        [96, 48],
        [32, 16, 8],
        [48, 24, 12],
        [64, 32, 16],
        [64, 48, 24],
        [96, 64, 32]
    ],
    "dropout": [0.01, 0.05, 0.10],
    "activation_name": ["relu", "leaky_relu", "elu"],
    "learning_rate": [0.1, 0.5, 1e-2, 5e-2, 5e-3, 1e-3, 5e-4, 1e-4],
    "batch_size": [16, 32, 64, 128],
    "weight_decay": [0.0, 1e-6, 1e-5, 1e-4, 1e-3],
    "loss_name": ["mse", "mae", "smoothl1"]
}

all_combinations = list(itertools.product(
    search_space["latent_dim"],
    range(len(search_space["hidden_dims"])),
    search_space["dropout"],
    search_space["activation_name"],
    search_space["learning_rate"],
    search_space["batch_size"],
    search_space["weight_decay"],
    search_space["loss_name"]
))

print(f"Total teórico de combinaciones: {len(all_combinations)}")

N_TRIALS = 660960
random.shuffle(all_combinations)
selected_trials = all_combinations[:N_TRIALS]

print(f"Se evaluarán {len(selected_trials)} trials.")

In [ ]:
# =========================================================
# 7. OPTIMIZACION
# =========================================================

resultados = []
histories = {}

for trial_num, trial in enumerate(selected_trials, start=1):
    latent_dim, hidden_idx, dropout, activation_name, lr, batch_size, weight_decay, loss_name = trial
    hidden_dims = search_space["hidden_dims"][hidden_idx]

    print("=" * 100)
    print(f"TRIAL {trial_num}/{N_TRIALS}")
    print(f"hidden_dims={hidden_dims}")
    print(f"latent_dim={latent_dim}")
    print(f"dropout={dropout}")
    print(f"activation={activation_name}")
    print(f"learning_rate={lr}")
    print(f"batch_size={batch_size}")
    print(f"weight_decay={weight_decay}")
    print(f"loss={loss_name}")

    train_dataset = EncuestaDataset(X_train)
    val_dataset = EncuestaDataset(X_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = AutoencoderFlexible(
        input_dim=X_train.shape[1],
        hidden_dims=hidden_dims,
        latent_dim=latent_dim,
        dropout=dropout,
        activation_name=activation_name
    ).to(DEVICE)

    criterion = get_loss_function(loss_name)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    model, history_df, best_val_loss = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        max_epochs=400,
        patience=40,
        min_delta=1e-6
    )

    Z_all = extract_latent(model, X_scaled)
    kmeans_metrics_df, best_k_row = evaluate_kmeans_on_embedding(Z_all, k_values=range(2, 7))

    if best_k_row is None:
        result = {
            "trial": trial_num,
            "hidden_dims": str(hidden_dims),
            "latent_dim": latent_dim,
            "dropout": dropout,
            "activation_name": activation_name,
            "learning_rate": lr,
            "batch_size": batch_size,
            "weight_decay": weight_decay,
            "loss_name": loss_name,
            "best_val_loss": best_val_loss,
            "best_k_by_silhouette": np.nan,
            "best_silhouette": np.nan,
            "best_davies_bouldin": np.nan,
            "best_calinski_harabasz": np.nan,
            "has_valid_clustering": 0
        }
    else:
        result = {
            "trial": trial_num,
            "hidden_dims": str(hidden_dims),
            "latent_dim": latent_dim,
            "dropout": dropout,
            "activation_name": activation_name,
            "learning_rate": lr,
            "batch_size": batch_size,
            "weight_decay": weight_decay,
            "loss_name": loss_name,
            "best_val_loss": best_val_loss,
            "best_k_by_silhouette": int(best_k_row["k"]),
            "best_silhouette": float(best_k_row["silhouette"]),
            "best_davies_bouldin": float(best_k_row["davies_bouldin"]),
            "best_calinski_harabasz": float(best_k_row["calinski_harabasz"]),
            "best_composite_score": float(best_k_row["composite_score"]),
            "has_valid_clustering": 1
        }

    resultados.append(result)
    histories[trial_num] = history_df

In [ ]:
# =========================================================
# 8. RANKING DE MODELOS
# =========================================================

resultados_df = pd.DataFrame(resultados)

resultados_df.to_excel("ranking_modelos_autoencoder_exhaustivo_todos.xlsx", index=False)

resultados_validos_df = resultados_df[resultados_df["has_valid_clustering"] == 1].copy()

if resultados_validos_df.empty:
    raise ValueError("Ningun trial produjo un clustering valido. Revisa el search space o el entrenamiento.")

resultados_validos_df = resultados_validos_df.sort_values(
    by=["best_val_loss", "best_composite_score"],
    ascending=[True, False]
).reset_index(drop=True)

resultados_validos_df.to_excel("ranking_modelos_autoencoder_exhaustivo_validos.xlsx", index=False)

print("\nTOP 15 modelos validos por validation loss:")
print(resultados_validos_df.head(15))

In [ ]:
# =========================================================
# 9. SELECCION DEL MEJOR MODELO FINAL
# =========================================================

top10 = resultados_validos_df.head(10).copy()
top10 = top10.sort_values(
    by=["best_composite_score"],
    ascending=[False]
).reset_index(drop=True)

mejor_fila = top10.iloc[0].to_dict()

print("\nMEJOR CONFIGURACION FINAL:")
print(mejor_fila)

best_hidden_dims = ast.literal_eval(mejor_fila["hidden_dims"])
best_latent_dim = int(mejor_fila["latent_dim"])
best_dropout = float(mejor_fila["dropout"])
best_activation = mejor_fila["activation_name"]
best_lr = float(mejor_fila["learning_rate"])
best_batch_size = int(mejor_fila["batch_size"])
best_weight_decay = float(mejor_fila["weight_decay"])
best_loss_name = mejor_fila["loss_name"]

In [ ]:
top10

In [ ]:
# =========================================================
# 10. GRAFICO DEL MEJOR TRIAL EN OPTIMIZACION
# =========================================================

best_trial_number = int(mejor_fila["trial"])
best_trial_history = histories[best_trial_number]

plt.figure(figsize=(8, 5))
plt.plot(best_trial_history["epoch"], best_trial_history["train_loss"], label="Train")
plt.plot(best_trial_history["epoch"], best_trial_history["val_loss"], label="Validation")
plt.title("Curvas del mejor trial en optimización")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("curvas_mejor_trial_optimizacion.png", dpi=200)
plt.close()

best_trial_history.to_excel("historial_mejor_trial_optimizacion.xlsx", index=False)

In [ ]:
# =========================================================
# 11. REENTRENAR EL MEJOR MODELO EN TODA LA DATA
# =========================================================

final_model, final_full_history = train_final_model_on_full_data(
    X_full=X_scaled,
    input_dim=X_scaled.shape[1],
    hidden_dims=best_hidden_dims,
    latent_dim=best_latent_dim,
    dropout=best_dropout,
    activation_name=best_activation,
    learning_rate=best_lr,
    weight_decay=best_weight_decay,
    batch_size=best_batch_size,
    loss_name=best_loss_name,
    max_epochs=500
)

final_full_history.to_excel("historial_mejor_modelo_full_data.xlsx", index=False)

plt.figure(figsize=(8, 5))
plt.plot(final_full_history["epoch"], final_full_history["full_loss"])
plt.title("Pérdida del mejor modelo reentrenado en toda la data")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.tight_layout()
plt.savefig("curva_mejor_modelo_full_data.png", dpi=200)
plt.close()

In [ ]:
# =========================================================
# 12. EMBEDDING FINAL
# =========================================================

Z_final = extract_latent(final_model, X_scaled)

latent_cols = [f"z{i+1}" for i in range(Z_final.shape[1])]
df_latent = pd.DataFrame(Z_final, columns=latent_cols)


In [ ]:
# =========================================================
# 13. K-MEANS FINAL
# =========================================================

kmeans_metrics_final, best_k_final_row = evaluate_kmeans_on_embedding(Z_final, k_values=range(2, 7))
best_k_final = int(best_k_final_row["k"])

print("\nMEJOR K FINAL POR SILHOUETTE:")
print(best_k_final_row)

kmeans_metrics_final.to_excel("metricas_kmeans_embedding_final.xlsx", index=False)

kmeans_final = KMeans(n_clusters=best_k_final, random_state=SEED, n_init=100)
cluster_labels_final = kmeans_final.fit_predict(Z_final)


In [ ]:
# =========================================================
# 14. BASE FINAL CON EMBEDDING Y CLUSTERS
# =========================================================

df_final = pd.concat(
    [
        df.reset_index(drop=True),
        df_latent.reset_index(drop=True)
    ],
    axis=1
)

df_final["cluster"] = cluster_labels_final

In [ ]:
# =========================================================
# 15. PERFIL DE PERCEPCIONES POR CLUSTER
#    (TRATANDO VARIABLES COMO ORDINALES/CATEGORICAS)
# =========================================================

def calcular_moda(series):
    moda = series.mode(dropna=True)
    if len(moda) == 0:
        return np.nan
    return moda.iloc[0]

def calcular_mediana_ordinal(series):
    s = series.dropna()
    if len(s) == 0:
        return np.nan
    return s.median()

writer_percepciones = pd.ExcelWriter(
    "perfil_clusters_percepciones_ordinal.xlsx",
    engine="openpyxl"
)

resumen_percepciones_rows = []

for var in VARIABLES_PERCEPCIONES:
    tabla_conteos = pd.crosstab(df_final["cluster"], df_final[var], dropna=False)

    tabla_porcentajes = pd.crosstab(
        df_final["cluster"],
        df_final[var],
        normalize="index",
        dropna=False
    ) * 100

    tabla_porcentajes = tabla_porcentajes.round(0).astype("Int64")

    sheet_conteos = f"{var}_n"[:31]
    sheet_pct = f"{var}_pct"[:31]

    tabla_conteos.to_excel(writer_percepciones, sheet_name=sheet_conteos)
    tabla_porcentajes.to_excel(writer_percepciones, sheet_name=sheet_pct)

    for cluster_id, subdf in df_final.groupby("cluster"):
        resumen_percepciones_rows.append({
            "variable": var,
            "cluster": cluster_id,
            "moda": calcular_moda(subdf[var]),
            "mediana": calcular_mediana_ordinal(subdf[var])
        })

resumen_percepciones_df = pd.DataFrame(resumen_percepciones_rows)
resumen_percepciones_df.to_excel(
    writer_percepciones,
    sheet_name="resumen_moda_mediana",
    index=False
)

writer_percepciones.close()

print("\nArchivo generado:")
print("- perfil_clusters_percepciones_ordinal.xlsx")

In [ ]:
# =========================================================
# 16. PERFIL SOCIODEMOGRAFICO POR CLUSTER
#    (USANDO CONTEOS Y PORCENTAJES)
# =========================================================

writer_sociodem = pd.ExcelWriter(
    "perfil_clusters_sociodemograficas_ordinal.xlsx",
    engine="openpyxl"
)

for var in VARIABLES_SOCIODEMOGRAFICAS_EXISTENTES:
    tabla_conteos = pd.crosstab(df_final["cluster"], df_final[var], dropna=False)

    tabla_porcentajes = pd.crosstab(
        df_final["cluster"],
        df_final[var],
        normalize="index",
        dropna=False
    ) * 100

    tabla_porcentajes = tabla_porcentajes.round(0).astype("Int64")

    sheet_conteos = f"{var}_n"[:31]
    sheet_pct = f"{var}_pct"[:31]

    tabla_conteos.to_excel(writer_sociodem, sheet_name=sheet_conteos)
    tabla_porcentajes.to_excel(writer_sociodem, sheet_name=sheet_pct)

writer_sociodem.close()

print("- perfil_clusters_sociodemograficas_ordinal.xlsx")

In [ ]:
# =========================================================
# 17. TABLA RESUMEN DE TAMANO DE CLUSTERS
# =========================================================

cluster_sizes = df_final["cluster"].value_counts().sort_index()
cluster_sizes_pct = (cluster_sizes / cluster_sizes.sum() * 100).round(0).astype(int)

tabla_tamano_clusters = pd.DataFrame({
    "cluster": cluster_sizes.index,
    "n": cluster_sizes.values,
    "pct": cluster_sizes_pct.values
})

tabla_tamano_clusters.to_excel("tamano_clusters.xlsx", index=False)

print("- tamano_clusters.xlsx")


In [ ]:
# =========================================================
# 18. VISUALIZACIONES CLAVE
# =========================================================

pca = PCA(n_components=2, random_state=SEED)
Z_2d = pca.fit_transform(Z_final)

df_plot = pd.DataFrame({
    "pc1": Z_2d[:, 0],
    "pc2": Z_2d[:, 1],
    "cluster": cluster_labels_final
})

plt.figure(figsize=(8, 6))
for c in sorted(df_plot["cluster"].unique()):
    tmp = df_plot[df_plot["cluster"] == c]
    plt.scatter(tmp["pc1"], tmp["pc2"], label=f"Cluster {c}", alpha=0.7)

plt.title("Clusters en espacio latente (PCA 2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("clusters_embedding_pca2d.png", dpi=200)
plt.close()

print("- clusters_embedding_pca2d.png")

In [ ]:
# =========================================================
# 19. TABLA LARGA DE PORCENTAJES DE PERCEPCIONES
#    (MUY UTIL PARA INTERPRETACION Y REPORTE)
# =========================================================

rows_long_percepciones = []

for var in VARIABLES_PERCEPCIONES:
    tabla_pct = pd.crosstab(
        df_final["cluster"],
        df_final[var],
        normalize="index",
        dropna=False
    ) * 100

    tabla_pct = tabla_pct.round(0)

    for cluster_id in tabla_pct.index:
        for nivel in tabla_pct.columns:
            rows_long_percepciones.append({
                "variable": var,
                "cluster": cluster_id,
                "nivel": nivel,
                "pct": int(tabla_pct.loc[cluster_id, nivel]) if pd.notnull(tabla_pct.loc[cluster_id, nivel]) else np.nan
            })

tabla_larga_percepciones = pd.DataFrame(rows_long_percepciones)
tabla_larga_percepciones.to_excel("tabla_larga_percepciones_pct.xlsx", index=False)

print("- tabla_larga_percepciones_pct.xlsx")

In [ ]:
# =========================================================
# 20. TABLA LARGA DE PORCENTAJES SOCIODEMOGRAFICOS
# =========================================================

rows_long_sociodem = []

for var in VARIABLES_SOCIODEMOGRAFICAS_EXISTENTES:
    tabla_pct = pd.crosstab(
        df_final["cluster"],
        df_final[var],
        normalize="index",
        dropna=False
    ) * 100

    tabla_pct = tabla_pct.round(0)

    for cluster_id in tabla_pct.index:
        for nivel in tabla_pct.columns:
            rows_long_sociodem.append({
                "variable": var,
                "cluster": cluster_id,
                "nivel": nivel,
                "pct": int(tabla_pct.loc[cluster_id, nivel]) if pd.notnull(tabla_pct.loc[cluster_id, nivel]) else np.nan
            })

tabla_larga_sociodem = pd.DataFrame(rows_long_sociodem)
tabla_larga_sociodem.to_excel("tabla_larga_sociodemograficas_pct.xlsx", index=False)

print("- tabla_larga_sociodemograficas_pct.xlsx")

In [ ]:
# =========================================================
# 21. ERROR DE RECONSTRUCCION FINAL POR PERSONA
# =========================================================

final_model.eval()
with torch.no_grad():
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE)
    X_recon, _ = final_model(X_tensor)
    X_recon = X_recon.cpu().numpy()

reconstruction_error = np.mean((X_scaled - X_recon) ** 2, axis=1)
df_final["reconstruction_error"] = reconstruction_error


In [ ]:
# =========================================================
# 22. EXPORTAR BASE FINAL
# =========================================================

df_final.to_excel("base_final_embeddings_clusters.xlsx", index=False)

torch.save(final_model.state_dict(), "mejor_autoencoder_final.pth")

scaler_min = pd.DataFrame({
    "variable": VARIABLES_PERCEPCIONES,
    "data_min_": scaler.data_min_,
    "data_max_": scaler.data_max_,
    "scale_": scaler.scale_,
    "min_": scaler.min_
})
scaler_min.to_excel("parametros_scaler.xlsx", index=False)

print("- base_final_embeddings_clusters.xlsx")
print("- mejor_autoencoder_final.pth")
print("- parametros_scaler.xlsx")

print("\nPROCESO TERMINADO CORRECTAMENTE.")

In [ ]:
latent_cols = [f"z{i+1}" for i in range(Z_final.shape[1])]
df_latent = pd.DataFrame(Z_final, columns=latent_cols)

df_final = pd.concat(
    [
        df.copy().reset_index(drop=True),
        df_latent.reset_index(drop=True)
    ],
    axis=1
)

df_final["cluster"] = cluster_labels_final

df_final.to_excel("base_original_con_embedding_y_cluster.xlsx", index=False)

In [ ]:
########################################### Mapas de calor

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
file_path = "ENADES2024.xlsx"

df = pd.read_excel(file_path)

print(df.head())

In [ ]:
dep_dict = {
1:"Amazonas",
2:"Ancash",
3:"Apurimac",
4:"Arequipa",
5:"Ayacucho",
6:"Cajamarca",
7:"Callao",
8:"Cusco",
9:"Huancavelica",
10:"Huanuco",
11:"Ica",
12:"Junin",
13:"La Libertad",
14:"Lambayeque",
15:"Lima",
16:"Loreto",
17:"Madre de Dios",
18:"Moquegua",
19:"Pasco",
20:"Piura",
21:"Puno",
22:"San Martin",
23:"Tacna",
24:"Tumbes",
25:"Ucayali"
}

In [ ]:
df["departamento"] = df["dep"].map(dep_dict)

print(df[["dep","departamento"]].head())

In [ ]:
tabla = df.groupby(["departamento","cluster"]).size().reset_index(name="n")

totales = df.groupby("departamento").size().reset_index(name="total_dep")

departamentos = df["departamento"].unique()
clusters = df["cluster"].unique()

index_completo = pd.MultiIndex.from_product(
    [departamentos, clusters],
    names=["departamento","cluster"]
)

tabla = (
    tabla.set_index(["departamento","cluster"])
    .reindex(index_completo, fill_value=0)
    .reset_index()
)

tabla = tabla.merge(totales, on="departamento", how="left")

tabla["prop"] = tabla["n"] / tabla["total_dep"]

In [ ]:
shapefile = "DEPARTAMENTOS_inei_geogpsperu_suyopomalia.shp"

mapa = gpd.read_file(shapefile)

print(mapa.columns)

In [ ]:
tabla["departamento"] = tabla["departamento"].str.upper()
mapa["NOMBDEP"] = mapa["NOMBDEP"].str.upper()

In [ ]:
clusters = sorted(tabla["cluster"].unique())

for c in clusters:

    datos_cluster = tabla[tabla["cluster"] == c]

    mapa_cluster = mapa.merge(
        datos_cluster,
        left_on="NOMBDEP",
        right_on="departamento",
        how="left"
    )

    fig, ax = plt.subplots(figsize=(10,8))

    mapa_cluster.plot(
        column="prop",
        cmap="YlOrRd",
        linewidth=0.8,
        edgecolor="black",
        legend=True,
        ax=ax
    )

    ax.set_title(f"Spatial Distribution of Cluster {c+1} Across Departments in Peru", fontsize=14)
    ax.axis("off")

    plt.show()

In [ ]:
for c in clusters:

    datos_cluster = tabla[tabla["cluster"] == c]

    mapa_cluster = mapa.merge(
        datos_cluster,
        left_on="NOMBDEP",
        right_on="departamento",
        how="left"
    )

    fig, ax = plt.subplots(figsize=(10,8))

    mapa_cluster.plot(
        column="prop",
        cmap="YlOrRd",
        linewidth=0.8,
        edgecolor="black",
        legend=True,
        ax=ax
    )

    ax.set_title(f"Cluster {c} - Perú", fontsize=14)
    ax.axis("off")

    plt.savefig(f"mapa_cluster_{c+1}_2024.png", dpi=300, bbox_inches="tight")
    plt.close()